In [6]:
import torch
import torch.nn as nn
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 20
NUM_FRAMES = 16
IMG_SIZE = 224
BATCH_SIZE = 8
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

print(f"Training on: {DEVICE}")

Training on: cpu


In [7]:
%run models.ipynb
%run preprocessing.ipynb

# model = ASLViT(num_classes=NUM_CLASSES).to(DEVICE)
model = ASLVideoMAE(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


print(f"Model Loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.norm.bias                                                    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias   

class to index mapping: {'accident': 0, 'bar': 1, 'bed': 2, 'before': 3, 'bowling': 4, 'call': 5, 'candy': 6, 'champion': 7, 'change': 8, 'check': 9, 'cold': 10, 'computer': 11, 'cool': 12, 'cousin': 13, 'deaf': 14, 'delay': 15, 'drink': 16, 'far': 17, 'go': 18, 'help': 19}
Extracted 16 frames from 00623.mp4
Frame shape: (240, 320, 3)
Total videos found: 227
Train: 158 | Val: 34 | Test: 35


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.norm.bias                                                    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias   

Model Loaded. Parameters: 86,429,204


In [8]:
# Training Loop
# from preprocessing import train_loader, val_loader, X_train, X_val
from tqdm import tqdm

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(NUM_EPOCHS):
    # training phase
    model.train()
    train_loss, train_correct = 0, 0

    for frames, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        frames, labels = frames.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    scheduler.step()

    # validation phase
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for frames, labels in tqdm(val_loader, desc=f"Validation"):
            frames, labels = frames.to(DEVICE), labels.to(DEVICE)
            outputs = model(frames)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    # calculate metrics
    train_loss_avg = train_loss / len(train_loader)
    val_loss_avg = val_loss / len(val_loader)
    train_acc = train_correct / len(X_train)
    val_acc = val_correct / len(X_val)

    train_losses.append(train_loss_avg)
    val_losses.append(val_loss_avg)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc:.4f}")    
        

Epoch 1/20:  10%|█         | 2/20 [04:28<40:18, 134.37s/it]


KeyboardInterrupt: 